# arrodeio-llm — SFT v3: better data, same recipe

Why: v2 stayed on-form but drifted off-topic ("cordel sobre o mar" -> a scene about a patroa).
Cause: the v2 prompts were guessed with keyword regexes + a *random* fallback theme, so many
pairs had an instruction that didn't describe its response.

v3 data (`scripts/make_sft_data_v2.py`): 797 chunks of 1 or 2 adjacent real stanzas, each with a
topic **written by hand by Claude** (`data/sft_v2_topics.jsonl`) — teacher labeling. (We tried
having Tucano 1.1B write the topics first; it mostly paraphrased the first verse.) Then the same LoRA SFT
(`scripts/sft.py`). The adapter goes to `model/cordel-sft-lora-v3` so v2 stays untouched.

**Runtime -> Change runtime type -> T4 GPU** before running. About 10-15 minutes.

### 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Code + libraries

In [ ]:
%cd /content
!rm -rf arrodeio-llm && git clone --depth 1 https://github.com/bsana1/arrodeio-llm.git
%cd /content/arrodeio-llm
!pip install -q -U transformers peft accelerate
!pip uninstall -q -y torchao

### 3. Build the v3 data

No GPU needed for this step; it just pairs each chunk with its written topic.

In [ ]:
!python scripts/make_sft_data_v2.py
!wc -l data/sft.jsonl data/sft_v2.jsonl

### 4. Train

One variable changed on purpose: the data. (`SFT_MAXLEN=512` only so 2-stanza responses aren't cut off.)

To also try the bigger base later, set `SFT_BASE=TucanoBR/Tucano-2b4-Instruct` and `SFT_BATCH=2`.

In [ ]:
import os
os.environ["SFT_DATA"] = "data/sft.jsonl,data/sft_v2.jsonl"
os.environ["SFT_OUT"] = "/content/arrodeio-llm/model/cordel-sft-lora-v3"
os.environ["SFT_MAXLEN"] = "512"
!python scripts/sft.py

### 5. Download the adapter

In [ ]:
import shutil
shutil.make_archive('/content/cordel-sft-lora-v3', 'zip', 'model/cordel-sft-lora-v3')
from google.colab import files
files.download('/content/cordel-sft-lora-v3.zip')

### 6. Back on your laptop

```bash
cd ~/src/arrodeio-llm
unzip ~/Downloads/cordel-sft-lora-v3.zip -d model/cordel-sft-lora-v3
```
Then compare against v2 on the same prompts before shipping.